In [1]:
# Cell 1
import sys
sys.path.append('../')
from src.compliance.alignment_engine import AlignmentEngine

In [2]:
# Cell 2
engine = AlignmentEngine(env_path='../.env')
print("Required activities:")
for act in engine.regulation['required_activities']:
    print(f"  - {act}")

Regulation loaded: 13 activities, 6 sequences, 4 conditions
Required activities:
  - O_Sent (mail and online)
  - O_Created
  - A_Accepted
  - A_Concept
  - O_Create Offer
  - A_Create Application
  - A_Complete


In [ ]:
# Cell 3 — Skip violation test
report_skip = engine.check_case('Application_1858876732')
print(f"=== SKIP VIOLATION TEST ===")
print(f"Tuân thủ: {report_skip.is_compliant}")
print(f"Fitness : {report_skip.fitness_score}")
for v in report_skip.violations:
    print(f"\n  [{v.severity.value.upper()}] {v.violation_type.value}")
    print(f"  {v.article_ref}: {v.description}")
    print(f"  Thực tế: {v.actual}")

In [ ]:
# Cell 4 — Frequency test sau khi sửa
report_freq = engine.check_case('Application_246097895')
print(f"=== FREQUENCY VIOLATION TEST ===")
print(f"Tuân thủ: {report_freq.is_compliant}")
print(f"Fitness : {report_freq.fitness_score}")
for v in report_freq.violations:
    print(f"\n  [{v.severity.value.upper()}] {v.violation_type.value}")
    print(f"  {v.article_ref}: {v.description}")
    print(f"  Thực tế: {v.actual}")
    print(f"  Evidence: call_count={v.evidence['call_count']}, "
          f"max={v.evidence['max_allowed']}")

In [ ]:
# Cell debug
trace = engine.trace_fetcher.get_trace('Application_246097895')

print(f"Tổng wf_events: {len(trace['wf_events'])}")
print(f"\nTất cả activity trong wf_events:")
activities = set(e['activity'] for e in trace['wf_events'])
for a in sorted(activities):
    print(f"  '{a}'")

print(f"\nCác event có 'Call' trong tên:")
for e in trace['wf_events']:
    if 'call' in e['activity'].lower() or 'Call' in e['activity']:
        print(f"  activity='{e['activity']}' | "
              f"lifecycle='{e['lifecycle']}' | "
              f"seq={e['seq_index']}")

In [ ]:
# Cell debug 2
call_schedules = [
    e for e in trace['wf_events']
    if e['activity'] == 'W_Call incomplete files'
    and e['lifecycle'] == 'schedule'
]
print(f"Số lần schedule (= số cuộc gọi): {len(call_schedules)}")

In [ ]:
# Cell debug 3 — tìm case có nhiều schedule nhất
import pandas as pd
df = pd.read_parquet('../data/processed/event_log_clean.parquet')

call_schedules = df[
    (df['activity'] == 'W_Call incomplete files') &
    (df['lifecycle'] == 'schedule')
].groupby('case_id').size()

print(f"Phân phối số lần schedule W_Call incomplete:")
print(f"  max    = {call_schedules.max()}")
print(f"  mean   = {call_schedules.mean():.1f}")
print(f"  median = {call_schedules.median()}")
print(f"\nTop 5 case nhiều schedule nhất:")
print(call_schedules.sort_values(ascending=False).head())
print(f"\nCase có >10 schedule: {(call_schedules > 10).sum()}")

In [ ]:
# Cell debug 4 — phân tích thời gian từ A_Incomplete đến A_Validating
incomplete_cases = df[df['activity'] == 'A_Incomplete']['case_id'].unique()
print(f"Case có A_Incomplete: {len(incomplete_cases)}")

durations = []
for cid in incomplete_cases[:500]:  # sample 500 case
    case_df = df[df['case_id'] == cid].sort_values('timestamp')
    incomplete_ts = case_df[
        case_df['activity'] == 'A_Incomplete'
    ]['timestamp'].values
    validating_ts = case_df[
        case_df['activity'] == 'A_Validating'
    ]['timestamp'].values

    if len(incomplete_ts) > 0 and len(validating_ts) > 0:
        t1 = pd.to_datetime(incomplete_ts[0])
        t2 = pd.to_datetime(validating_ts[-1])
        days = (t2 - t1).days
        if days >= 0:
            durations.append(days)

import numpy as np
durations = pd.Series(durations)
print(f"\nThời gian A_Incomplete → A_Validating (ngày):")
print(f"  min    = {durations.min()}")
print(f"  p50    = {durations.median():.0f}")
print(f"  p75    = {durations.quantile(0.75):.0f}")
print(f"  p90    = {durations.quantile(0.90):.0f}")
print(f"  p95    = {durations.quantile(0.95):.0f}")
print(f"  max    = {durations.max()}")

In [4]:
# Cell debug 5 — tìm case xử lý incomplete > 8 ngày
import pandas as pd
df = pd.read_parquet('../data/processed/event_log_clean.parquet')

incomplete_cases = df[df['activity'] == 'A_Incomplete']['case_id'].unique()
slow_cases = []

for cid in incomplete_cases:
    case_df = df[df['case_id'] == cid].sort_values('timestamp')
    t1 = case_df[case_df['activity'] == 'A_Incomplete']['timestamp']
    t2 = case_df[case_df['activity'] == 'A_Validating']['timestamp']
    if len(t1) > 0 and len(t2) > 0:
        days = (pd.to_datetime(t2.iloc[-1]) -
                pd.to_datetime(t1.iloc[0])).days
        if days > 8:
            slow_cases.append((cid, days))

slow_cases.sort(key=lambda x: x[1], reverse=True)
print(f"Case xử lý incomplete > 8 ngày: {len(slow_cases)}")
print(f"\nTop 5:")
for cid, days in slow_cases[:5]:
    print(f"  {cid}: {days} ngày")

Case xử lý incomplete > 8 ngày: 889

Top 5:
  Application_35950487: 60 ngày
  Application_1099835060: 55 ngày
  Application_845569937: 55 ngày
  Application_774441889: 51 ngày
  Application_797343379: 51 ngày


In [5]:
# Cell debug 6 — tìm case gửi offer chậm > 1 ngày
offer_delays = []
offer_cases = df[df['activity'] == 'O_Created']['case_id'].unique()

for cid in offer_cases:
    case_df = df[df['case_id'] == cid].sort_values('timestamp')
    t1 = case_df[case_df['activity'] == 'O_Created']['timestamp']
    t2 = case_df[case_df['activity'].isin([
        'O_Sent (mail and online)', 'O_Sent (online only)'
    ])]['timestamp']
    if len(t1) > 0 and len(t2) > 0:
        days = (pd.to_datetime(t2.iloc[0]) -
                pd.to_datetime(t1.iloc[0])).days
        if days > 1:
            offer_delays.append((cid, days))

offer_delays.sort(key=lambda x: x[1], reverse=True)
print(f"Case gửi offer chậm > 1 ngày: {len(offer_delays)}")
print(f"\nTop 5:")
for cid, days in offer_delays[:5]:
    print(f"  {cid}: {days} ngày")

Case gửi offer chậm > 1 ngày: 91

Top 5:
  Application_373627092: 29 ngày
  Application_667158687: 21 ngày
  Application_681572147: 18 ngày
  Application_2144842406: 15 ngày
  Application_1882054030: 13 ngày


In [6]:
# Cell test incomplete handling
report_inc = engine.check_case('Application_35950487')
print(f"=== INCOMPLETE HANDLING TEST ===")
print(f"Tuân thủ: {report_inc.is_compliant}")
print(f"Fitness : {report_inc.fitness_score}")
for v in report_inc.violations:
    print(f"\n  [{v.severity.value.upper()}] {v.violation_type.value}")
    print(f"  {v.article_ref}: {v.description}")
    print(f"  Thực tế: {v.actual}")

=== INCOMPLETE HANDLING TEST ===
Tuân thủ: False
Fitness : 0.85

  [MEDIUM] temporal
  Article 7(1): Xử lý hồ sơ thiếu quá 60 ngày, vượt ngưỡng 8 ngày
  Thực tế: A_Incomplete lúc 2016-08-26 09:28:19.461000+00:00 → A_Validating lúc 2016-10-25 11:11:59.789000+00:00 = 60 ngày


In [7]:
# Cell test offer sending delay
report_off = engine.check_case('Application_373627092')
print(f"=== OFFER SENDING DELAY TEST ===")
print(f"Tuân thủ: {report_off.is_compliant}")
print(f"Fitness : {report_off.fitness_score}")
for v in report_off.violations:
    print(f"\n  [{v.severity.value.upper()}] {v.violation_type.value}")
    print(f"  {v.article_ref}: {v.description}")
    print(f"  Thực tế: {v.actual}")

=== OFFER SENDING DELAY TEST ===
Tuân thủ: False
Fitness : 0.85

  [MEDIUM] temporal
  Article 5(1): Offer tạo xong nhưng gửi sau 29 ngày, vượt ngưỡng 1 ngày
  Thực tế: O_Created lúc 2016-09-01 11:21:24.634000+00:00 → O_Sent lúc 2016-09-30 13:03:07.116000+00:00 = 29 ngày


In [ ]:
# Cell 5 — Temporal violation test
report_temp = engine.check_case('Application_1639833431')
print(f"=== TEMPORAL VIOLATION TEST ===")
print(f"Tuân thủ: {report_temp.is_compliant}")
print(f"Fitness : {report_temp.fitness_score}")
for v in report_temp.violations:
    print(f"\n  [{v.severity.value.upper()}] {v.violation_type.value}")
    print(f"  {v.article_ref}: {v.description}")
    print(f"  Thực tế: {v.actual}")

In [ ]:
# Cell 6 — Debug frequency checker
import pandas as pd
df = pd.read_parquet('../data/processed/event_log_clean.parquet')

case_id = 'Application_246097895'
wf_calls = df[
    (df['case_id'] == case_id) &
    (df['activity'] == 'W_Call incomplete files')
]
print(f"Tổng W_Call incomplete files: {len(wf_calls)}")
print(f"\nPhân phối lifecycle:")
print(wf_calls['lifecycle'].value_counts())

In [8]:
# Cell final — chạy toàn bộ và lưu kết quả
import os
reports_all = engine.check_all(
    save_path='../data/processed/compliance_results.json'
)

Bắt đầu kiểm tra 28512 case...
  1000/28512 — tuân thủ: 832, vi phạm: 168, lỗi: 0
  2000/28512 — tuân thủ: 1673, vi phạm: 327, lỗi: 0
  3000/28512 — tuân thủ: 2531, vi phạm: 469, lỗi: 0
  4000/28512 — tuân thủ: 3402, vi phạm: 598, lỗi: 0
  5000/28512 — tuân thủ: 4239, vi phạm: 761, lỗi: 0
  6000/28512 — tuân thủ: 5104, vi phạm: 896, lỗi: 0
  7000/28512 — tuân thủ: 5968, vi phạm: 1032, lỗi: 0
  8000/28512 — tuân thủ: 6827, vi phạm: 1173, lỗi: 0
  9000/28512 — tuân thủ: 7681, vi phạm: 1319, lỗi: 0
  10000/28512 — tuân thủ: 8523, vi phạm: 1477, lỗi: 0
  11000/28512 — tuân thủ: 9377, vi phạm: 1623, lỗi: 0
  12000/28512 — tuân thủ: 10238, vi phạm: 1762, lỗi: 0
  13000/28512 — tuân thủ: 11082, vi phạm: 1918, lỗi: 0
  14000/28512 — tuân thủ: 11894, vi phạm: 2106, lỗi: 0
  15000/28512 — tuân thủ: 12757, vi phạm: 2243, lỗi: 0
  16000/28512 — tuân thủ: 13624, vi phạm: 2376, lỗi: 0
  17000/28512 — tuân thủ: 14466, vi phạm: 2534, lỗi: 0
  18000/28512 — tuân thủ: 15324, vi phạm: 2676, lỗi: 0
  1900

In [9]:
# Cell summary — thống kê tổng thể
import pandas as pd
from collections import Counter

summary = [{
    'case_id'         : r.case_id,
    'is_compliant'    : r.is_compliant,
    'fitness_score'   : r.fitness_score,
    'num_violations'  : len(r.violations),
    'num_high'        : r.violation_summary['high'],
    'num_medium'      : r.violation_summary['medium'],
    'application_type': r.application_type,
    'requested_amount': r.requested_amount,
} for r in reports_all]

df_summary = pd.DataFrame(summary)
df_summary.to_csv(
    '../data/processed/compliance_summary.csv',
    index=False
)

total      = len(df_summary)
compliant  = df_summary['is_compliant'].sum()
violated   = total - compliant

print(f"{'='*45}")
print(f"TỔNG KẾT COMPLIANCE CHECKING")
print(f"{'='*45}")
print(f"Tổng case         : {total:,}")
print(f"Tuân thủ          : {compliant:,} ({compliant/total*100:.1f}%)")
print(f"Vi phạm           : {violated:,} ({violated/total*100:.1f}%)")
print(f"Fitness score TB  : {df_summary['fitness_score'].mean():.3f}")
print(f"\nPhân phối vi phạm theo loại:")

vtype_counter = Counter()
severity_counter = Counter()
for r in reports_all:
    for v in r.violations:
        vtype_counter[v.violation_type.value] += 1
        severity_counter[v.severity.value] += 1

for vtype, cnt in vtype_counter.most_common():
    print(f"  {vtype:<12}: {cnt:,} vi phạm")

print(f"\nPhân phối theo mức độ:")
for sev, cnt in severity_counter.most_common():
    print(f"  {sev:<8}: {cnt:,}")

print(f"\nPhân phối fitness score:")
print(df_summary['fitness_score'].describe().round(3))

TỔNG KẾT COMPLIANCE CHECKING
Tổng case         : 28,512
Tuân thủ          : 24,221 (85.0%)
Vi phạm           : 4,291 (15.0%)
Fitness score TB  : 0.973

Phân phối vi phạm theo loại:
  temporal    : 4,205 vi phạm
  skip        : 361 vi phạm
  order       : 119 vi phạm

Phân phối theo mức độ:
  medium  : 4,205
  high    : 480

Phân phối fitness score:
count    28512.000
mean         0.973
std          0.071
min          0.400
25%          1.000
50%          1.000
75%          1.000
max          1.000
Name: fitness_score, dtype: float64


In [10]:
# Cell close
engine.close()